# Train coffee-leaf multi-label disease classifiers

Notebook này chứa toàn bộ dataset contract, EDA, augmentation, training loop, early stopping và evaluation cho bộ Kaggle Coffee leaf diseases. Mỗi lá có ba nhãn độc lập miner, rust, phoma; healthy được suy ra khi không có bệnh nào vượt ngưỡng.

DVC thực thi notebook bằng Papermill và so sánh bốn backbone pretrained: EfficientNetV2-S, RegNetY-3.2GF, ConvNeXt-Tiny và DenseNet121. Notebook lưu checkpoint, JSON/CSV metrics, EDA, training curves, biểu đồ so sánh và confusion matrices; pipeline.py chỉ chọn checkpoint sau khi train.

In [ ]:
params_path = "params.yaml"


In [ ]:
from collections.abc import Sequence
from pathlib import Path
from typing import Any
import csv
import json
import os
import time

project_root = Path.cwd().resolve()
if not (project_root / "pipeline.py").exists():
    project_root = project_root.parent
if not (project_root / "pipeline.py").exists():
    raise RuntimeError("Run this notebook from the CoffeeLeaf-AI repository")
os.chdir(project_root)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from IPython.display import Markdown, display
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    hamming_loss,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
)
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms

from pipeline import (
    IMAGENET_MEAN,
    IMAGENET_STD,
    ROOT,
    build_classifier,
    load_config,
    project_path,
    reset_dir,
    seed_everything,
    write_csv,
    write_json,
)

print(f"Repository: {ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
class LeafCropDataset:
    """Kaggle image/mask pairs with three independent disease targets."""

    def __init__(
        self,
        root: Path,
        manifest_path: Path,
        split: str,
        disease_classes: Sequence[str],
        transform: Any,
        mask_background: bool,
        background_rgb: Sequence[int],
        fill_rgb: Sequence[int],
    ) -> None:
        self.root = root.resolve()
        self.split = split
        self.disease_classes = list(disease_classes)
        self.transform = transform
        self.mask_background = mask_background
        self.background_rgb = np.asarray(background_rgb, dtype=np.uint8)
        self.fill_rgb = np.asarray(fill_rgb, dtype=np.uint8)
        self.samples: list[tuple[Path, Path, tuple[float, ...]]] = []
        with manifest_path.open("r", encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            required = {"image", "mask", "split", *self.disease_classes}
            missing = required.difference(reader.fieldnames or [])
            if missing:
                raise ValueError(f"{manifest_path} is missing columns: {sorted(missing)}")
            for row_number, row in enumerate(reader, start=2):
                if row["split"] != split:
                    continue
                image = (self.root / row["image"]).resolve()
                mask = (self.root / row["mask"]).resolve()
                if self.root not in image.parents or self.root not in mask.parents:
                    raise ValueError(f"{manifest_path}:{row_number}: unsafe asset path")
                if not image.is_file() or not mask.is_file():
                    raise FileNotFoundError(
                        f"{manifest_path}:{row_number}: missing image or mask"
                    )
                targets = tuple(float(int(row[name])) for name in self.disease_classes)
                if any(value not in {0.0, 1.0} for value in targets):
                    raise ValueError(f"{manifest_path}:{row_number}: targets must be 0/1")
                self.samples.append((image, mask, targets))
        if not self.samples:
            raise ValueError(f"No classification samples for split {split!r}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int) -> tuple[Any, Any]:
        image_path, mask_path, targets = self.samples[index]
        with Image.open(image_path) as source:
            image = source.convert("RGB")
        with Image.open(mask_path) as source:
            mask = source.convert("RGB")
        if image.size != mask.size:
            raise ValueError(f"Image/mask dimensions differ: {image_path} and {mask_path}")
        if self.mask_background:
            image_array = np.asarray(image, dtype=np.uint8).copy()
            mask_array = np.asarray(mask, dtype=np.uint8)
            foreground = np.any(mask_array != self.background_rgb, axis=2)
            image_array[~foreground] = self.fill_rgb
            image = Image.fromarray(image_array, mode="RGB")
        return self.transform(image), torch.tensor(targets, dtype=torch.float32)


def classifier_transforms(
    image_size: int,
    fill_rgb: Sequence[int],
    augmentation: dict[str, Any],
) -> tuple[Any, Any]:
    crop_scale = tuple(float(value) for value in augmentation["crop_scale"])
    crop_ratio = tuple(float(value) for value in augmentation["crop_ratio"])
    jitter = float(augmentation["color_jitter"])
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(
            image_size, scale=crop_scale, ratio=crop_ratio, antialias=True
        ),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomVerticalFlip(0.5),
        transforms.RandomRotation(
            float(augmentation["rotation_degrees"]), fill=tuple(fill_rgb)
        ),
        transforms.ColorJitter(
            brightness=jitter, contrast=jitter, saturation=jitter, hue=min(0.05, jitter / 4)
        ),
        transforms.RandomApply(
            [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.5))],
            p=float(augmentation["gaussian_blur_probability"]),
        ),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(
            p=float(augmentation["random_erasing_probability"]),
            scale=(0.02, 0.08),
            ratio=(0.5, 2.0),
            value="random",
        ),
    ])
    evaluation_transform = transforms.Compose([
        transforms.Resize((image_size, image_size), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train_transform, evaluation_transform


def evaluate_classifier(
    model: Any,
    loader: Any,
    device: Any,
    threshold: float,
    healthy_label: str,
    criterion: Any = None,
) -> dict[str, Any]:
    model.eval()
    targets: list[list[int]] = []
    probabilities: list[list[float]] = []
    total_loss = 0.0
    with torch.inference_mode():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            logits = model(inputs)
            if criterion is not None:
                total_loss += float(criterion(logits, labels).item()) * labels.size(0)
            probabilities.extend(logits.sigmoid().cpu().tolist())
            targets.extend(labels.int().cpu().tolist())

    target_array = np.asarray(targets, dtype=np.int64)
    probability_array = np.asarray(probabilities, dtype=np.float32)
    prediction_array = (probability_array >= threshold).astype(np.int64)
    healthy_targets = (target_array.sum(axis=1) == 0).astype(np.int64)[:, None]
    healthy_predictions = (prediction_array.sum(axis=1) == 0).astype(np.int64)[:, None]
    output_targets = np.concatenate([healthy_targets, target_array], axis=1)
    output_predictions = np.concatenate([healthy_predictions, prediction_array], axis=1)
    output_classes = [healthy_label, *loader.dataset.disease_classes]
    precision, recall, f1, support = precision_recall_fscore_support(
        output_targets, output_predictions, average=None, zero_division=0
    )
    _, _, micro_f1, _ = precision_recall_fscore_support(
        output_targets, output_predictions, average="micro", zero_division=0
    )
    disease_f1 = f1[1:]
    return {
        "loss": total_loss / max(1, len(target_array)),
        "accuracy": float(accuracy_score(target_array, prediction_array)),
        "hamming_loss": float(hamming_loss(target_array, prediction_array)),
        "precision": precision.tolist(),
        "recall": recall.tolist(),
        "f1": f1.tolist(),
        "support": support.astype(int).tolist(),
        "macro_f1": float(np.mean(f1)),
        "disease_macro_f1": float(np.mean(disease_f1)),
        "micro_f1": float(micro_f1),
        "output_classes": output_classes,
        "targets": target_array.tolist(),
        "predictions": prediction_array.tolist(),
        "output_targets": output_targets.tolist(),
        "output_predictions": output_predictions.tolist(),
    }


In [ ]:
config = load_config(params_path)
seed = int(config["seed"])
seed_everything(seed)
settings = config["classification"]
augmentation_settings = dict(settings["augmentation"])
data_settings = config["data"]
disease_classes = list(data_settings["disease_classes"])
healthy_label = str(data_settings["healthy_label"])
output_classes = [healthy_label, *disease_classes]
threshold = float(config["deployment"]["classifier_confidence"])
data_root = project_path(data_settings["processed_dir"]) / "classification"
manifest_path = data_root / "labels.csv"
contract_path = data_root / "dataset.json"
if not manifest_path.is_file() or not contract_path.is_file():
    raise RuntimeError(
        "Prepared classification data is missing. Run dvc repro prepare first."
    )
contract = json.loads(contract_path.read_text(encoding="utf-8"))
if contract.get("classification_mode") != "multilabel":
    raise ValueError("Prepared classification contract must be multilabel")
if list(contract.get("disease_classes", [])) != disease_classes:
    raise ValueError("Prepared disease class order differs from params.yaml")
if str(contract.get("healthy_label")) != healthy_label:
    raise ValueError("Prepared healthy label differs from params.yaml")
background_rgb = list(contract["mask_background_rgb"])
fill_rgb = list(data_settings["mask_fill_rgb"])
mask_background = bool(data_settings["mask_background"])
metrics_dir = ROOT / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps(settings, indent=2))
print(f"Outputs: {output_classes}; disease threshold={threshold}")
print("Augmentation:", json.dumps(augmentation_settings, indent=2))


## Exploratory data analysis

EDA dùng manifest đã được stage `prepare` tạo ra. Các biểu đồ chỉ mô tả split và nhãn; test labels không được dùng để chọn hyperparameter hay checkpoint.

In [ ]:
with manifest_path.open("r", encoding="utf-8-sig", newline="") as stream:
    manifest_rows = list(csv.DictReader(stream))

split_names = ("train", "val", "test")
split_counts = {split: 0 for split in split_names}
label_counts = {
    split: {name: 0 for name in output_classes}
    for split in split_names
}
combination_counts: dict[str, dict[str, int]] = {
    split: {} for split in split_names
}
train_vectors: list[list[int]] = []

for row in manifest_rows:
    split = row["split"]
    if split not in split_counts:
        continue
    diseases = [int(row[name]) for name in disease_classes]
    healthy = int(sum(diseases) == 0)
    output_vector = [healthy, *diseases]
    split_counts[split] += 1
    for name, value in zip(output_classes, output_vector):
        label_counts[split][name] += value
    combination = healthy_label if healthy else "+".join(
        name for name, value in zip(disease_classes, diseases) if value
    )
    combination_counts[split][combination] = (
        combination_counts[split].get(combination, 0) + 1
    )
    if split == "train":
        train_vectors.append(output_vector)

train_matrix = np.asarray(train_vectors, dtype=np.int64)
co_occurrence = (train_matrix.T @ train_matrix).astype(int)
eda_payload = {
    "classification_mode": "multilabel",
    "split_counts": split_counts,
    "label_counts": label_counts,
    "combination_counts": combination_counts,
    "train_co_occurrence": {
        row_name: {
            column_name: int(co_occurrence[row_index, column_index])
            for column_index, column_name in enumerate(output_classes)
        }
        for row_index, row_name in enumerate(output_classes)
    },
}
write_json(metrics_dir / "classification_eda.json", eda_payload)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
split_colors = ["#2563eb", "#f59e0b", "#10b981"]
axes[0, 0].bar(split_names, [split_counts[name] for name in split_names], color=split_colors)
axes[0, 0].set_title("Images per split")
axes[0, 0].set_ylabel("Images")
for index, split in enumerate(split_names):
    axes[0, 0].text(index, split_counts[split], str(split_counts[split]), ha="center", va="bottom")

positions = np.arange(len(output_classes))
bar_width = 0.24
for offset, split in enumerate(split_names):
    values = [label_counts[split][name] for name in output_classes]
    axes[0, 1].bar(
        positions + (offset - 1) * bar_width,
        values,
        width=bar_width,
        label=split,
        color=split_colors[offset],
    )
axes[0, 1].set_xticks(positions, output_classes)
axes[0, 1].set_title("Label support by split")
axes[0, 1].set_ylabel("Positive labels")
axes[0, 1].legend()

train_combinations = sorted(
    combination_counts["train"].items(), key=lambda item: item[1]
)
axes[1, 0].barh(
    [name for name, _ in train_combinations],
    [count for _, count in train_combinations],
    color="#7c3aed",
)
axes[1, 0].set_title("Train label combinations")
axes[1, 0].set_xlabel("Images")

sns.heatmap(
    co_occurrence,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    xticklabels=output_classes,
    yticklabels=output_classes,
    cbar=False,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Train label co-occurrence")
fig.suptitle("Coffee leaf classification EDA", fontsize=16, fontweight="bold")
fig.tight_layout()
eda_plot_path = metrics_dir / "classification_eda.png"
fig.savefig(eda_plot_path, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)
print(f"Saved EDA metrics: {metrics_dir / 'classification_eda.json'}")
print(f"Saved EDA plot: {eda_plot_path}")

## Train and evaluate candidates

Checkpoint được chọn hoàn toàn bằng validation Macro F1. Test metrics chỉ được tính sau khi đã nạp lại checkpoint tốt nhất.

In [ ]:
output_dir = reset_dir(ROOT / "models" / "classifiers")
metrics_by_candidate: dict[str, dict[str, Any]] = {}
comparison_rows: list[dict[str, Any]] = []
history_by_candidate: dict[str, list[dict[str, Any]]] = {}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
worker_count = 0 if os.name == "nt" else int(settings["workers"])
print(f"DataLoader workers: {worker_count}")

for candidate, candidate_settings in settings["candidates"].items():
    print(f"\n=== Training classifier: {candidate} ===")
    architecture = str(candidate_settings.get("architecture", candidate))
    image_size = int(candidate_settings["image_size"])
    candidate_batch_size = int(
        candidate_settings.get("batch_size", settings["batch_size"])
    )
    train_transform, eval_transform = classifier_transforms(
        image_size, fill_rgb, augmentation_settings
    )
    dataset_args = {
        "root": data_root,
        "manifest_path": manifest_path,
        "disease_classes": disease_classes,
        "mask_background": mask_background,
        "background_rgb": background_rgb,
        "fill_rgb": fill_rgb,
    }
    train_dataset = LeafCropDataset(
        split="train", transform=train_transform, **dataset_args
    )
    val_dataset = LeafCropDataset(
        split="val", transform=eval_transform, **dataset_args
    )
    test_dataset = LeafCropDataset(
        split="test", transform=eval_transform, **dataset_args
    )
    generator = torch.Generator().manual_seed(seed)
    loader_args = {
        "batch_size": candidate_batch_size,
        "num_workers": worker_count,
        "pin_memory": device.type == "cuda",
    }
    train_loader = DataLoader(
        train_dataset, shuffle=True, generator=generator, **loader_args
    )
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)
    test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

    model, head = build_classifier(
        architecture,
        len(disease_classes),
        float(candidate_settings["dropout"]),
        int(candidate_settings["unfreeze_blocks"]),
        bool(settings["pretrained"]),
    )
    model.to(device)
    total_params_m = sum(parameter.numel() for parameter in model.parameters()) / 1e6
    trainable_params_m = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    ) / 1e6
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    target_matrix = np.asarray(
        [targets for _, _, targets in train_dataset.samples], dtype=np.float32
    )
    positive_counts = target_matrix.sum(axis=0)
    if np.any(positive_counts == 0):
        raise ValueError("Every disease must have positive training samples")
    negative_counts = len(train_dataset) - positive_counts
    pos_weight = torch.tensor(
        negative_counts / positive_counts, dtype=torch.float32, device=device
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    head_ids = {id(parameter) for parameter in head.parameters()}
    backbone_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad and id(parameter) not in head_ids
    ]
    parameter_groups = [
        {"params": list(head.parameters()), "lr": float(settings["head_lr"])}
    ]
    if backbone_parameters:
        parameter_groups.insert(
            0,
            {
                "params": backbone_parameters,
                "lr": float(settings["backbone_lr"]),
            },
        )
    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=float(settings["weight_decay"]),
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=int(settings["epochs"]), eta_min=1e-6
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    target_path = output_dir / f"{candidate}.pt"
    history: list[dict[str, Any]] = []
    best_macro_f1 = -1.0
    best_epoch = 0
    stale_epochs = 0
    smoothing = float(settings["label_smoothing"])
    candidate_started = time.perf_counter()

    for epoch in range(1, int(settings["epochs"]) + 1):
        model.train()
        train_loss = 0.0
        train_exact_matches = 0
        train_total = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            smoothed_labels = labels * (1.0 - smoothing) + 0.5 * smoothing
            with torch.autocast(
                device_type=device.type, dtype=torch.float16, enabled=use_amp
            ):
                logits = model(inputs)
                loss = criterion(logits, smoothed_labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            predictions = (logits.sigmoid() >= threshold).int()
            train_exact_matches += int(
                (predictions == labels.int()).all(dim=1).sum().item()
            )
            train_loss += float(loss.item()) * labels.size(0)
            train_total += labels.size(0)
        scheduler.step()
        validation = evaluate_classifier(
            model, val_loader, device, threshold, healthy_label, criterion
        )
        epoch_row = {
            "epoch": epoch,
            "train_loss": train_loss / max(1, train_total),
            "train_exact_match_accuracy": train_exact_matches / max(1, train_total),
            "val_loss": validation["loss"],
            "val_macro_f1": validation["macro_f1"],
            "val_disease_macro_f1": validation["disease_macro_f1"],
        }
        history.append(epoch_row)
        print(
            f"epoch={epoch:03d} train_loss={epoch_row['train_loss']:.4f} "
            f"val_loss={epoch_row['val_loss']:.4f} "
            f"val_f1={epoch_row['val_macro_f1']:.4f}"
        )
        if validation["macro_f1"] > best_macro_f1 + 1e-6:
            best_macro_f1 = validation["macro_f1"]
            best_epoch = epoch
            stale_epochs = 0
            torch.save(
                {
                    "model_name": architecture,
                    "candidate_name": candidate,
                    "state_dict": model.state_dict(),
                    "classification_mode": "multilabel",
                    "disease_classes": disease_classes,
                    "healthy_label": healthy_label,
                    "decision_threshold": threshold,
                    "image_size": image_size,
                    "dropout": float(candidate_settings["dropout"]),
                    "unfreeze_blocks": int(candidate_settings["unfreeze_blocks"]),
                },
                target_path,
            )
        else:
            stale_epochs += 1
            if stale_epochs >= int(settings["patience"]):
                print(f"Early stopping {candidate} at epoch {epoch}")
                break

    history_by_candidate[candidate] = history
    write_csv(output_dir / f"{candidate}_history.csv", history)
    checkpoint = torch.load(target_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    validation_result = evaluate_classifier(
        model, val_loader, device, threshold, healthy_label, criterion
    )
    test_result = evaluate_classifier(
        model, test_loader, device, threshold, healthy_label, criterion
    )
    matrices = multilabel_confusion_matrix(
        np.asarray(test_result["output_targets"]),
        np.asarray(test_result["output_predictions"]),
    ).tolist()

    sample, _ = test_dataset[0]
    sample = sample.unsqueeze(0).to(device)
    model.eval()
    with torch.inference_mode():
        for _ in range(5):
            model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        for _ in range(30):
            model(sample)
        if device.type == "cuda":
            torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - started) * 1000 / 30
    training_seconds = time.perf_counter() - candidate_started
    peak_vram_mb = (
        torch.cuda.max_memory_allocated(device) / (1024 * 1024)
        if device.type == "cuda"
        else 0.0
    )

    validation_per_class = {
        name: {
            "precision": float(validation_result["precision"][index]),
            "recall": float(validation_result["recall"][index]),
            "f1": float(validation_result["f1"][index]),
            "support": int(validation_result["support"][index]),
        }
        for index, name in enumerate(output_classes)
    }
    test_per_class = {
        name: {
            "precision": float(test_result["precision"][index]),
            "recall": float(test_result["recall"][index]),
            "f1": float(test_result["f1"][index]),
            "support": int(test_result["support"][index]),
            "confusion_matrix": matrices[index],
        }
        for index, name in enumerate(output_classes)
    }
    validation_summary = {
        "accuracy": validation_result["accuracy"],
        "hamming_loss": validation_result["hamming_loss"],
        "macro_f1": validation_result["macro_f1"],
        "disease_macro_f1": validation_result["disease_macro_f1"],
        "micro_f1": validation_result["micro_f1"],
        "min_class_recall": min(
            item["recall"] for item in validation_per_class.values()
        ),
    }
    test_summary = {
        "accuracy": test_result["accuracy"],
        "hamming_loss": test_result["hamming_loss"],
        "macro_f1": test_result["macro_f1"],
        "disease_macro_f1": test_result["disease_macro_f1"],
        "micro_f1": test_result["micro_f1"],
        "min_class_recall": min(
            item["recall"] for item in test_per_class.values()
        ),
    }
    operational = {
        "latency_ms": latency_ms,
        "fps": 1000.0 / max(latency_ms, 1e-9),
        "size_mb": target_path.stat().st_size / (1024 * 1024),
        "total_params_m": total_params_m,
        "trainable_params_m": trainable_params_m,
        "peak_vram_mb": peak_vram_mb,
        "training_seconds": training_seconds,
        "best_epoch": best_epoch,
        "epochs_trained": len(history),
        "image_size": image_size,
        "batch_size": candidate_batch_size,
    }
    write_json(
        output_dir / f"{candidate}_report.json",
        {
            "classification_mode": "multilabel",
            "decision_threshold": threshold,
            "validation": validation_summary,
            "test": test_summary,
            "operational": operational,
            "validation_per_class": validation_per_class,
            "test_per_class": test_per_class,
        },
    )
    values = {
        "accuracy": validation_summary["accuracy"],
        "hamming_loss": validation_summary["hamming_loss"],
        "macro_f1": validation_summary["macro_f1"],
        "disease_macro_f1": validation_summary["disease_macro_f1"],
        "micro_f1": validation_summary["micro_f1"],
        "min_class_recall": validation_summary["min_class_recall"],
        "test_accuracy": test_summary["accuracy"],
        "test_hamming_loss": test_summary["hamming_loss"],
        "test_macro_f1": test_summary["macro_f1"],
        "test_disease_macro_f1": test_summary["disease_macro_f1"],
        "test_micro_f1": test_summary["micro_f1"],
        "test_min_class_recall": test_summary["min_class_recall"],
        **operational,
        "decision_threshold": threshold,
        "validation_per_class": validation_per_class,
        "test_per_class": test_per_class,
    }
    metrics_by_candidate[candidate] = values
    comparison_rows.append(
        {
            "candidate": candidate,
            **{
                key: values[key]
                for key in (
                    "accuracy",
                    "macro_f1",
                    "disease_macro_f1",
                    "min_class_recall",
                    "test_accuracy",
                    "test_macro_f1",
                    "test_disease_macro_f1",
                    "test_min_class_recall",
                    "test_hamming_loss",
                    "latency_ms",
                    "fps",
                    "size_mb",
                    "total_params_m",
                    "trainable_params_m",
                    "peak_vram_mb",
                    "training_seconds",
                    "best_epoch",
                    "epochs_trained",
                    "image_size",
                    "batch_size",
                )
            },
        }
    )
    print(
        f"{candidate}: val_macro_f1={values['macro_f1']:.4f}, "
        f"test_macro_f1={values['test_macro_f1']:.4f}, "
        f"latency={latency_ms:.2f} ms, size={values['size_mb']:.2f} MB"
    )
    del model, head, optimizer, scheduler, scaler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

write_json(
    metrics_dir / "classifiers.json",
    {
        "classification_mode": "multilabel",
        "disease_classes": disease_classes,
        "healthy_label": healthy_label,
        "decision_threshold": threshold,
        "candidates": metrics_by_candidate,
    },
)
write_csv(metrics_dir / "classifiers.csv", comparison_rows)

## Curves and final comparison

Các biểu đồ dưới đây được lưu dưới `metrics/`, nên DVC và DagsHub có thể hiển thị lại sau mỗi experiment.

In [ ]:
candidate_names = list(metrics_by_candidate)
palette = sns.color_palette("tab10", n_colors=len(candidate_names))

fig, axes = plt.subplots(
    len(candidate_names), 2, figsize=(14, 4 * len(candidate_names)), squeeze=False
)
for row_index, candidate in enumerate(candidate_names):
    history = history_by_candidate[candidate]
    epochs = [item["epoch"] for item in history]
    axes[row_index, 0].plot(
        epochs, [item["train_loss"] for item in history], label="train loss"
    )
    axes[row_index, 0].plot(
        epochs, [item["val_loss"] for item in history], label="validation loss"
    )
    axes[row_index, 0].set_title(f"{candidate}: loss")
    axes[row_index, 0].set_xlabel("Epoch")
    axes[row_index, 0].legend()

    axes[row_index, 1].plot(
        epochs,
        [item["val_macro_f1"] for item in history],
        label="validation macro F1",
    )
    axes[row_index, 1].plot(
        epochs,
        [item["val_disease_macro_f1"] for item in history],
        label="validation disease macro F1",
    )
    axes[row_index, 1].set_ylim(0.0, 1.02)
    axes[row_index, 1].set_title(f"{candidate}: validation F1")
    axes[row_index, 1].set_xlabel("Epoch")
    axes[row_index, 1].legend()
fig.suptitle("Classifier training curves", fontsize=16, fontweight="bold")
fig.tight_layout()
training_plot_path = metrics_dir / "classifier_training_curves.png"
fig.savefig(training_plot_path, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

comparison_specs = [
    ("macro_f1", "Validation Macro F1", True),
    ("test_macro_f1", "Test Macro F1", True),
    ("latency_ms", "Latency (ms/image)", False),
    ("size_mb", "Checkpoint size (MB)", False),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for axis, (metric_name, title, unit_interval) in zip(axes.flat, comparison_specs):
    values = [metrics_by_candidate[name][metric_name] for name in candidate_names]
    bars = axis.bar(candidate_names, values, color=palette)
    axis.set_title(title)
    axis.tick_params(axis="x", rotation=20)
    if unit_interval:
        axis.set_ylim(0.0, 1.02)
    for bar, value in zip(bars, values):
        label = f"{value:.3f}" if unit_interval else f"{value:.1f}"
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            label,
            ha="center",
            va="bottom",
            fontsize=9,
        )
fig.suptitle("Classifier candidate comparison", fontsize=16, fontweight="bold")
fig.tight_layout()
comparison_plot_path = metrics_dir / "classifier_comparison.png"
fig.savefig(comparison_plot_path, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, axes = plt.subplots(
    len(candidate_names),
    len(output_classes),
    figsize=(3.5 * len(output_classes), 3.0 * len(candidate_names)),
    squeeze=False,
)
for row_index, candidate in enumerate(candidate_names):
    for column_index, class_name in enumerate(output_classes):
        matrix = np.asarray(
            metrics_by_candidate[candidate]["test_per_class"][class_name][
                "confusion_matrix"
            ]
        )
        axis = axes[row_index, column_index]
        sns.heatmap(
            matrix,
            annot=True,
            fmt="d",
            cmap="Blues",
            cbar=False,
            xticklabels=["negative", "positive"],
            yticklabels=["negative", "positive"],
            ax=axis,
        )
        axis.set_title(f"{candidate}\n{class_name}")
        axis.set_xlabel("Predicted")
        axis.set_ylabel("Actual")
fig.suptitle("Independent test confusion matrices", fontsize=16, fontweight="bold")
fig.tight_layout()
confusion_plot_path = metrics_dir / "classifier_confusion_matrices.png"
fig.savefig(confusion_plot_path, dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)

best_quality_candidate = max(
    candidate_names, key=lambda name: metrics_by_candidate[name]["macro_f1"]
)
table_lines = [
    "| Candidate | Val Macro F1 | Test Macro F1 | Test exact match | Latency ms | Size MB | Params M | Best epoch |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for candidate in candidate_names:
    values = metrics_by_candidate[candidate]
    table_lines.append(
        f"| {candidate} | {values['macro_f1']:.4f} | "
        f"{values['test_macro_f1']:.4f} | {values['test_accuracy']:.4f} | "
        f"{values['latency_ms']:.2f} | {values['size_mb']:.2f} | "
        f"{values['total_params_m']:.2f} | {int(values['best_epoch'])} |"
    )
display(Markdown("\n".join(table_lines)))
print(f"Best validation Macro F1: {best_quality_candidate}")
print(f"Saved plots: {training_plot_path}, {comparison_plot_path}, {confusion_plot_path}")

In [ ]:
result = json.loads(
    (metrics_dir / "classifiers.json").read_text(encoding="utf-8")
)
result